# CNN + LSTM — Identify Fruit from a Sequence of Images
**Dataset:** Fake 4-frame fruit image sequences (no download needed)
**Input:** 4 image frames | **Output:** Apple / Banana / Orange

> CNN extracts color features from each frame. LSTM remembers those features across all frames. Like reading a flipbook to identify a fruit!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, Conv2D, MaxPooling2D, Flatten, LSTM, Dense, Dropout, Input

np.random.seed(42)

# ── Create Fruit Image Sequences ──────────────────────────
# Each sample = 4 frames of 8x8 RGB image
def make_sequences(label, n=40):
    seqs = []
    for _ in range(n):
        frames = []
        for _ in range(4):
            img = np.random.uniform(0, 0.1, (8, 8, 3))
            if label == 0:    # Apple = RED
                img[:,:,0] = np.random.uniform(0.8, 1.0, (8,8))
            elif label == 1:  # Banana = YELLOW
                img[:,:,0] = np.random.uniform(0.8, 1.0, (8,8))
                img[:,:,1] = np.random.uniform(0.8, 1.0, (8,8))
            else:             # Orange
                img[:,:,0] = np.random.uniform(0.8, 1.0, (8,8))
                img[:,:,1] = np.random.uniform(0.4, 0.6, (8,8))
            frames.append(img)
        seqs.append(frames)
    return np.array(seqs)   # shape: (n, 4, 8, 8, 3)

X = np.vstack([make_sequences(0), make_sequences(1), make_sequences(2)])
y = np.array([0]*40 + [1]*40 + [2]*40)

# ── Train / Test Split ─────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'X shape: {X.shape}  |  Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# ── Show a Sample Sequence ────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i in range(4):
    axes[i].imshow(X[0][i])   # Apple sequence
    axes[i].set_title(f'Frame {i+1}'); axes[i].axis('off')
plt.suptitle('Apple — 4 Frame Sequence')
plt.show()

In [ ]:
# ── Build CNN + LSTM ──────────────────────────────────────
# TimeDistributed = apply same CNN to every frame independently
# LSTM            = learn patterns across the 4 frames
model = Sequential([
    Input(shape=(4, 8, 8, 3)),                          # 4 frames of 8x8 RGB
    TimeDistributed(Conv2D(16, (3,3), activation='relu')),
    TimeDistributed(MaxPooling2D((2,2))),
    TimeDistributed(Flatten()),
    LSTM(32),
    Dropout(0.2),
    Dense(3, activation='softmax')                      # 3 fruits
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────
history = model.fit(X_train, y_train, epochs=30, batch_size=16,
                    validation_data=(X_test, y_test), verbose=0)

print(f'Train Accuracy: {history.history["accuracy"][-1]*100:.1f}%')
print(f'Test  Accuracy: {history.history["val_accuracy"][-1]*100:.1f}%')

In [ ]:
# ── Accuracy Plot ─────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'],     label='Train')
plt.plot(history.history['val_accuracy'], label='Test', linestyle='--')
plt.title('CNN+LSTM — Training Accuracy over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                              display_labels=['Apple','Banana','Orange'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — CNN+LSTM Fruit Classifier')
plt.show()

In [ ]:
# ── Predict on All 3 Fruit Types ─────────────────────────
fruit_names = ['Apple', 'Banana', 'Orange']
fig, axes = plt.subplots(3, 4, figsize=(10, 8))

for row, idx in enumerate([0, 40, 80]):
    pred    = model.predict(X[idx:idx+1], verbose=0).argmax()
    actual  = fruit_names[y[idx]]
    predicted = fruit_names[pred]
    for col in range(4):
        axes[row][col].imshow(X[idx][col])
        axes[row][col].set_title(f'Frame {col+1}', fontsize=8)
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f'True: {actual}\nPred: {predicted}', fontsize=9)

plt.suptitle('CNN+LSTM — Fruit Sequence Predictions')
plt.tight_layout()
plt.show()